In [4]:
pip install -q faiss-cpu sentence-transformers rank-bm25 bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.6 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 111.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 106.1 MB/s eta 0:00:0000:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

print("Hugging Face login successful")

Hugging Face login successful


# CHUNK CLEANING FOR Submission 8

In [1]:
# page splitted
# LOAD KNOWLEDGE BASE
# ======================================================

INPUT_PATH = "/kaggle/input/competitions/are-you-sure-llm-is-enough-intra-cuet-ml-contest-2-0/Dataset/Knowledge_Base.txt"

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"KB loaded.")
print(f"Total chars: {len(raw_text):,}")

# ======================================================
# SPLIT BY PAGE TITLE MARKER
# ======================================================

marker = " - উইকিপিডিয়া"

parts = raw_text.split(marker)

pages = []

# rebuild pages correctly
for i in range(len(parts) - 1):

    title = parts[i].split("\n")[-1].strip()

    page = (
        title
        + marker
        + parts[i + 1]
    )

    pages.append(page)

print("\n" + "=" * 70)
print("PAGE STATS")
print("=" * 70)

print(f"Total pages found: {len(pages):,}")

print("\nFirst 20 page titles:")
print("-" * 70)

for i, page in enumerate(pages[:20]):
    title = page.split("\n")[0].strip()
    print(f"{i+1:>2}. {title}")

# ======================================================
# CHECK PASHUPATI
# ======================================================

keyword = "পশুপতি প্রসাদ মাহাতো"

found = False

for idx, page in enumerate(pages):

    title = page.split("\n")[0]

    if keyword in title:

        found = True

        print("\n" + "=" * 70)
        print("PASHUPATI PAGE FOUND")
        print("=" * 70)

        print(f"Page index: {idx:,}")

        print("\nTitle:")
        print(title)

        print("\nPreview:")
        print("-" * 70)
        print(page[:1500])

        break

if not found:
    print("\nPashupati page NOT found.")

KB loaded.
Total chars: 56,516,686

PAGE STATS
Total pages found: 6,993

First 20 page titles:
----------------------------------------------------------------------
 1. ﻿মোঃ আব্দুল মুক্তাদির - উইকিপিডিয়া
 2. সোলো সিকোয়া - উইকিপিডিয়া
 3. বিলি নিউহাম - উইকিপিডিয়া
 4. শিখধর্মের সমালোচনা - উইকিপিডিয়া
 5. নিকিতা কোন্তিনি - উইকিপিডিয়া
 6. শহীদ স্মৃতি বিদ্যাপীঠ - উইকিপিডিয়া
 7. জাহান্নাম থেকে চিঠি - উইকিপিডিয়া
 8. স্পেনীয় ফুটবল লিগ পদ্ধতি - উইকিপিডিয়া
 9. জার্মান লেবার ফ্রন্ট - উইকিপিডিয়া
10. হাতুড়ি ও নেহাই (রণচাতুরি) - উইকিপিডিয়া
11. কাবেরী (অভিনেত্রী) - উইকিপিডিয়া
12. গীবত - উইকিপিডিয়া
13. কমুনিকাসিওনেস ফুটবল ক্লাব - উইকিপিডিয়া
14. ২০২২–২৩ বিসিএল ওয়ানডে - উইকিপিডিয়া
15. ডোনাভান ফ্রেবার্গ - উইকিপিডিয়া
16. বাইনিউরাল বিট - উইকিপিডিয়া
17. দ্য ব্লাডলাইন - উইকিপিডিয়া
18. আসামের শহরের তালিকা - উইকিপিডিয়া
19. ইয়েন্স কাজুস্ত - উইকিপিডিয়া
20. তখত শ্রী দমদমা সাহিব - উইকিপিডিয়া

PASHUPATI PAGE FOUND
Page index: 2,817

Title:
পশুপতি প্রসাদ মাহাতো - উইকিপিডিয়া

Preview:
-------

In [3]:
# MAIN CLEANING

import re

# ======================================================
# CONFIG
# ======================================================

INPUT_PATH = "/kaggle/input/competitions/are-you-sure-llm-is-enough-intra-cuet-ml-contest-2-0/Dataset/Knowledge_Base.txt"
OUTPUT_PATH = "/kaggle/working/Knowledge_Base_cleaned_v3_page_dhore_dhore.txt"

# ======================================================
# HARD-CODED BLOCKS
# ======================================================

HEADER_BLOCK = """বিষয়বস্তুতে চলুন
প্রধান মেনু
প্রধান মেনু
পার্শ্বদণ্ডে নিন
লুকান
পরিভ্রমণ
প্রধান পাতা
সম্প্রদায়ের প্রবেশদ্বার
সম্প্রদায়ের আলোচনাসভা
সাম্প্রতিক পরিবর্তন
অজানা যেকোনো পাতা
সাহায্য
অনুসন্ধান
অনুসন্ধান
অবয়ব
দান করুন
অ্যাকাউন্ট তৈরি করুন
প্রবেশ করুন
নিজস্ব সরঞ্জামসমূহ
দান করুন
অ্যাকাউন্ট তৈরি করুন
প্রবেশ করুন
পরিচ্ছেদসমূহ
পার্শ্বদণ্ডে নিন
লুকান"""

FOOTER_BLOCK = """এই পাতাটি
পার্সোইড
দিয়ে রেন্ডার করা হয়েছে।
লেখাগুলো
ক্রিয়েটিভ কমন্স অ্যাট্রিবিউশন/শেয়ার-আলাইক লাইসেন্সের
আওতাভুক্ত; এর সাথে বাড়তি শর্ত প্রযোজ্য হতে পারে। এই সাইট ব্যবহার করার মাধ্যমে আপনি এর
ব্যবহারের শর্তাবলী
ও
গোপনীয়তা নীতির
সাথে সম্মত হচ্ছেন। উইকিপিডিয়া® একটি অলাভজনক সংস্থা, যা
উইকিমিডিয়া ফাউন্ডেশনের
একটি নিবন্ধিত ট্রেডমার্ক।
গোপনীয়তার নীতি
উইকিপিডিয়া বৃত্তান্ত
দাবিত্যাগ
আচরণবিধি
উন্নয়নকারী
পরিসংখ্যান
কুকির বিবৃতি
মোবাইল সংস্করণ
অনুসন্ধান
অনুসন্ধান
সূচিপত্র টগল করুন"""

TOOLS_BLOCK = """নিবন্ধ
আলোচনা
বাংলা
পড়ুন
সম্পাদনা
ইতিহাস দেখুন
সরঞ্জাম
সরঞ্জাম
পার্শ্বদণ্ডে নিন
লুকান
কার্য
পড়ুন
সম্পাদনা
ইতিহাস দেখুন
সাধারণ
সংযোগকারী পৃষ্ঠাসমূহ
সম্পর্কিত পরিবর্তন
আপলোড করুন
স্থায়ী সংযোগ
পাতার তথ্য
এই নিবন্ধটি উদ্ধৃত করুন
সংক্ষিপ্ত ইউআরএল নিন
সংক্ষিপ্ত ইউআরএল
পূর্বের পার্সারে স্যুইচ করুন
মুদ্রণ/রপ্তানি
বই তৈরি করুন
PDF  ডাউনলোড
মুদ্রণযোগ্য সংস্করণ
অন্যান্য প্রকল্পে
উইকিউপাত্ত আইটেম
অবয়ব
পার্শ্বদণ্ডে নিন
লুকান"""

# ======================================================
# GLOBAL COUNTERS
# ======================================================

header_removed = 0
footer_removed = 0
tools_removed = 0

# ======================================================
# CLEAN SINGLE PAGE
# ======================================================

def clean_page(page):
    global header_removed
    global footer_removed
    global tools_removed

    # -------------------------
    # Header
    # -------------------------
    header_count = page.count(HEADER_BLOCK)

    if header_count:
        page = page.replace(HEADER_BLOCK, "")
        header_removed += header_count

    # -------------------------
    # Footer
    # -------------------------
    footer_count = page.count(FOOTER_BLOCK)

    if footer_count:
        page = page.replace(FOOTER_BLOCK, "")
        footer_removed += footer_count

    # -------------------------
    # Tools
    # -------------------------
    tools_count = page.count(TOOLS_BLOCK)

    if tools_count:
        page = page.replace(TOOLS_BLOCK, "")
        tools_removed += tools_count

    # -------------------------
    # Whitespace cleanup
    # -------------------------
    page = re.sub(r'\n{3,}', '\n\n', page)

    return page.strip()


# ======================================================
# LOAD KB
# ======================================================

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

original_size = len(raw_text)

print("=" * 70)
print("KB LOADED")
print("=" * 70)

print(f"Original size: {original_size:,} chars")

# ======================================================
# SPLIT INTO PAGES
# ======================================================

marker = " - উইকিপিডিয়া"

parts = raw_text.split(marker)

pages = []

for i in range(len(parts) - 1):

    title = parts[i].split("\n")[-1].strip()

    page = (
        title
        + marker
        + parts[i + 1]
    )

    pages.append(page)

print(f"Pages found: {len(pages):,}")

# ======================================================
# CLEAN PAGE-BY-PAGE
# ======================================================

print("\n" + "=" * 70)
print("CLEANING PAGES")
print("=" * 70)

cleaned_pages = []

for i, page in enumerate(pages):

    cleaned = clean_page(page)
    cleaned_pages.append(cleaned)

    if (i + 1) % 5000 == 0:
        print(f"Processed {i+1:,} pages")

# ======================================================
# REJOIN
# ======================================================

cleaned_text = "\n\n".join(cleaned_pages)

cleaned_size = len(cleaned_text)

reduction = (
    (original_size - cleaned_size)
    / original_size
) * 100

# ======================================================
# SMOKE TEST
# ======================================================

print("\n" + "=" * 70)
print("SMOKE TEST")
print("=" * 70)

test_pages = [
    "পশুপতি প্রসাদ মাহাতো",
    "সোলো সিকোয়া",
    "মোঃ আব্দুল মুক্তাদির"
]

for name in test_pages:
    print(
        f"{name}:",
        "FOUND" if name in cleaned_text else "MISSING"
    )

# ======================================================
# REPORT
# ======================================================

print("\n" + "=" * 70)
print("CLEANING REPORT")
print("=" * 70)

print(f"Pages processed        : {len(pages):,}")
print(f"Header blocks removed  : {header_removed:,}")
print(f"Footer blocks removed  : {footer_removed:,}")
print(f"Tools blocks removed   : {tools_removed:,}")

print("-" * 70)

total_removed = (
    header_removed
    + footer_removed
    + tools_removed
)

print(f"Total blocks removed   : {total_removed:,}")

print("\nSIZE STATISTICS")
print(f"Original size          : {original_size:,}")
print(f"Cleaned size           : {cleaned_size:,}")
print(f"Reduction              : {reduction:.2f}%")

# ======================================================
# SAVE
# ======================================================

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

print(f"Saved to:\n{OUTPUT_PATH}")

KB LOADED
Original size: 56,516,686 chars
Pages found: 6,993

CLEANING PAGES
Processed 5,000 pages

SMOKE TEST
পশুপতি প্রসাদ মাহাতো: FOUND
সোলো সিকোয়া: FOUND
মোঃ আব্দুল মুক্তাদির: FOUND

CLEANING REPORT
Pages processed        : 6,993
Header blocks removed  : 6,812
Footer blocks removed  : 6,812
Tools blocks removed   : 3,901
----------------------------------------------------------------------
Total blocks removed   : 17,525

SIZE STATISTICS
Original size          : 56,516,686
Cleaned size           : 49,379,142
Reduction              : 12.63%

DONE
Saved to:
/kaggle/working/Knowledge_Base_cleaned_v3_page_dhore_dhore.txt


In [5]:
# ==========================================================
# IMPROVED WIKI KB CHUNKER
# Sentence-safe + overlap-safe + clean starts
# ==========================================================

import re
import pickle

# ----------------------------------------------------------
# PATHS
# ----------------------------------------------------------

KB_PATH = "/kaggle/working/Knowledge_Base_cleaned_v3_page_dhore_dhore.txt"

CHUNKS_PATH = "/kaggle/working/chunks_recommended2.pkl"

# ----------------------------------------------------------
# LOAD KB
# ----------------------------------------------------------

print("Reading KB...")

with open(KB_PATH, "r", encoding="utf-8", errors="ignore") as f:
    kb_text = f.read()

# Remove BOM
kb_text = kb_text.lstrip('\ufeff')

print("KB size:", len(kb_text))


# ==========================================================
# IMPROVED CHUNKER
# ==========================================================

def chunk_text_better(
    text,
    chunk_size=800,
    overlap=150
):
    """
    Better chunker for Bangla wiki KB

    Features:
    - sentence-safe endings
    - paragraph-aware splitting
    - preserved overlap
    - avoids broken word starts
    """

    # Normalize repeated newlines
    text = re.sub(r'\n+', '\n', text)

    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:

        # --------------------------------------------------
        # Initial end
        # --------------------------------------------------
        end = min(start + chunk_size, text_len)

        # --------------------------------------------------
        # End of document
        # --------------------------------------------------
        if end == text_len:

            final_chunk = text[start:].strip()

            if len(final_chunk) > 50:
                chunks.append(final_chunk)

            break

        # --------------------------------------------------
        # PRIORITY 1: paragraph boundary
        # --------------------------------------------------
        paragraph_end = text.rfind(
            "\n",
            start + 300,
            end
        )

        # --------------------------------------------------
        # PRIORITY 2: sentence boundary
        # --------------------------------------------------
        sentence_end = max(
            text.rfind("।", start + 300, end),
            text.rfind(".", start + 300, end)
        )

        # --------------------------------------------------
        # Choose best boundary
        # --------------------------------------------------
        boundary = max(
            paragraph_end,
            sentence_end
        )

        # fallback
        if boundary == -1:
            boundary = end
        else:
            boundary += 1

        # --------------------------------------------------
        # Create chunk
        # --------------------------------------------------
        chunk = text[start:boundary].strip()

        if len(chunk) > 50:
            chunks.append(chunk)

        # --------------------------------------------------
        # Preserve overlap
        # --------------------------------------------------
        candidate_start = max(
            0,
            boundary - overlap
        )

        # --------------------------------------------------
        # FIX:
        # Move to clean boundary
        # Avoid broken words like:
        # "া বিশ্ববিদ্যালয়"
        # --------------------------------------------------
        newline_pos = text.find(
            "\n",
            candidate_start,
            min(candidate_start + 200, text_len)
        )

        sentence_pos = text.find(
            "।",
            candidate_start,
            min(candidate_start + 200, text_len)
        )

        possible_positions = [
            p for p in
            [newline_pos, sentence_pos]
            if p != -1
        ]

        if possible_positions:
            start = min(possible_positions) + 1
        else:
            start = candidate_start

    return chunks


# ==========================================================
# GENERATE CHUNKS
# ==========================================================

print("Generating chunks...")

chunks = chunk_text_better(
    kb_text,
    chunk_size=800,
    overlap=150
)

print(f"Total chunks: {len(chunks):,}")

# ----------------------------------------------------------
# SAVE
# ----------------------------------------------------------

with open(CHUNKS_PATH, "wb") as f:
    pickle.dump(chunks, f)

print(f"\n✓ Saved chunks to:")
print(CHUNKS_PATH)

# ==========================================================
# CHUNK STATISTICS
# ==========================================================

lengths = [len(c) for c in chunks]

print("\n" + "=" * 70)
print("CHUNK STATISTICS")
print("=" * 70)

print(f"Min length : {min(lengths):,}")
print(f"Max length : {max(lengths):,}")
print(f"Avg length : {sum(lengths)/len(lengths):.1f}")

# ==========================================================
# PREVIEW
# ==========================================================

print("\n" + "=" * 70)
print("SAMPLE CHUNKS")
print("=" * 70)

for i in range(min(4, len(chunks))):

    print(f"\nCHUNK {i+1}")
    print("-" * 50)

    print(chunks[i][:700])

Reading KB...
KB size: 49379141
Generating chunks...
Total chunks: 75,819

✓ Saved chunks to:
/kaggle/working/chunks_recommended2.pkl

CHUNK STATISTICS
Min length : 291
Max length : 800
Avg length : 772.8

SAMPLE CHUNKS

CHUNK 1
--------------------------------------------------
মোঃ আব্দুল মুক্তাদির - উইকিপিডিয়া
সূচনা
১
জীবনের প্রথমার্ধ
২
পেশা
৩
মৃত্যু এবং উত্তরাধিকার
৪
তথ্যসূত্র
৫
বহিঃসংযোগ
সূচিপত্র টগল করুন
মোঃ আব্দুল মুক্তাদির
১টি ভাষা
English
আন্তঃউইকি সংযোগ সম্পাদনা
উইকিপিডিয়া, মুক্ত বিশ্বকোষ থেকে
শহীদ
ড.
মোহাম্মদ আবদুল মুক্তাদির
জন্ম
(
১৯৪০-০২-১৯
)
১৯ ফেব্রুয়ারি ১৯৪০
পশ্চিমপাড়া,
সিলাম
,
সিলেট জেলা
,
ব্রিটিশ ভারত
মৃত্যু
২৫ মার্চ ১৯৭১
(1971-03-25)
(বয়স
৩১)
পূর্ব পাকিস্তান
মৃতদেহ আবিস্কার
ইকবাল হল (বর্তমানে সার্জেন্ট জহুরুল হক হল)
সমাধি
পল্টন কবরস্থান
নাগরিকত্ব
ব্রিটিশ ভারত
(১৯৪০-১৯৪৭),
পাকিস্তান
(১৯৪৭-১৯৭১)
শিক্ষা
চকবাজার সরকারী প্রাথমিক বিদ্যালয়, সিলাম পিএল জুনিয়র স্কুল, রাজা জি.সি. উচ্চ বিদ্যালয়, সিলেট সরকারি কলেজ
মাতৃশিক্ষায়তন
ঢাকা বিশ্ববিদ্যালয়
,
ল

CHUNK 2
----------

In [3]:
# ==========================================================
# STEP 2: GENERATE BGE-M3 EMBEDDINGS
# (FOR NEW CHUNKS)
# ==========================================================

import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

# ----------------------------------------------------------
# PATHS
# ----------------------------------------------------------

CHUNKS_PATH = "/kaggle/working/chunks_recommended2.pkl"

BGE_EMBED_PATH = (
    "/kaggle/working/bge_m3_embeddings_recommended2.npy"
)

# ----------------------------------------------------------
# LOAD CHUNKS
# ----------------------------------------------------------

print("Loading chunks...")

with open(CHUNKS_PATH, "rb") as f:
    chunks = pickle.load(f)

print("Chunks:", len(chunks))

# ----------------------------------------------------------
# LOAD BGE-M3
# ----------------------------------------------------------

print("Loading BGE-M3...")

embed_model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cuda"
)

# ----------------------------------------------------------
# GENERATE EMBEDDINGS
# ----------------------------------------------------------

print("Encoding chunks...")

chunk_embeddings = embed_model.encode(
    chunks,
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# ----------------------------------------------------------
# SAVE EMBEDDINGS
# ----------------------------------------------------------

np.save(
    BGE_EMBED_PATH,
    chunk_embeddings
)

print("Saved BGE-M3 embeddings!")
print("Embedding shape:", chunk_embeddings.shape)
print("Saved to:", BGE_EMBED_PATH)

Loading chunks...
Chunks: 75819
Loading BGE-M3...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Encoding chunks...


Batches:   0%|          | 0/4739 [00:00<?, ?it/s]

Saved BGE-M3 embeddings!
Embedding shape: (75819, 1024)
Saved to: /kaggle/working/bge_m3_embeddings_recommended2.npy
